# 02 Stage 1 Logistic Regression

Train and evaluate Logistic Regression for all three Stage 1 barrier targets.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.split_scale import split_and_scale
from src.models.logistic_regression import train_logistic

print('Project root:', PROJECT_ROOT)

Project root: B:\PBCS\Major Project\BarrierLens_MP_G25_P48


In [ ]:
X = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'X_features.csv')
y_household = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'y_household.csv').squeeze('columns')
y_logistic = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'y_logistic.csv').squeeze('columns')
y_facility = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'y_facility.csv').squeeze('columns')

print('X shape:', X.shape)
print('Household distribution:', y_household.value_counts().to_dict())

X shape: (706, 54)
Household distribution: {1: 353, 0: 353}


In [ ]:
def evaluate_binary(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
    }


def train_one_target(X_df, y, target_name):
    combo = pd.concat([X_df, y.rename(f'target_{target_name}')], axis=1)
    X_train, X_test, y_train, y_test, _ = split_and_scale(combo, f'target_{target_name}')
    feature_names = X_df.columns.tolist()

    model, coefs = train_logistic(X_train, y_train, feature_names, target_name)
    metrics = evaluate_binary(model, X_test, y_test)

    cv_model = LogisticRegression(solver='lbfgs', max_iter=500, random_state=42, C=model.C)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(cv_model, X_df, y, cv=cv, scoring='roc_auc')

    return model, coefs, metrics, cv_scores

In [ ]:
results = []

for y_series, target_name in [
    (y_household, 'household'),
    (y_logistic, 'logistic'),
    (y_facility, 'facility'),
]:
    model, coefs, holdout_metrics, cv_scores = train_one_target(X, y_series, target_name)
    row = {
        'Model': 'Logistic Regression',
        'Target': target_name,
        **holdout_metrics,
        'CV AUC Mean': round(cv_scores.mean(), 4),
        'CV AUC Std': round(cv_scores.std(), 4),
    }
    results.append(row)
    print(f"\n{target_name.upper()}\n", row)

results_df = pd.DataFrame(results)
results_df

target_household - Train: 564 rows | Test: 142 rows


Best C for household: {'C': 1.0}
                       Feature  Coefficient  OddsRatio
51      State/UT_Uttar Pradesh     1.648377   5.198534
53        State/UT_West Bengal     0.884898   2.422736
38            State/UT_Manipur     0.875064   2.399028
37         State/UT_Maharastra     0.812560   2.253669
31    State/UT_Jammu & Kashmir     0.747186   2.111050
23              State/UT_Bihar     0.634581   1.886232
29            State/UT_Haryana     0.526663   1.693272
21  State/UT_Arunachal Pradesh     0.514484   1.672775
50            State/UT_Tripura     0.486724   1.626978
47             State/UT_Sikkim     0.474830   1.607741


c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stab

c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



FACILITY
 {'Model': 'Logistic Regression', 'Target': 'facility', 'Accuracy': 0.8099, 'ROC-AUC': 0.8693, 'Precision': 0.8056, 'Recall': 0.8169, 'F1-Score': 0.8112, 'CV AUC Mean': 0.8791, 'CV AUC Std': 0.0373}


c:\users\pbcsh\appdata\local\programs\python\python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Target,Accuracy,ROC-AUC,Precision,Recall,F1-Score,CV AUC Mean,CV AUC Std
0,Logistic Regression,household,0.9437,0.9849,0.9200,0.9718,0.9452,0.9532,0.0183
1,Logistic Regression,logistic,0.8239,0.9224,0.8286,0.8169,0.8227,0.9177,0.0210
2,Logistic Regression,facility,0.8099,0.8693,0.8056,0.8169,0.8112,0.8791,0.0373


In [ ]:
out_dir = PROJECT_ROOT / 'outputs' / 'stage1_results'
out_dir.mkdir(parents=True, exist_ok=True)

results_path = out_dir / 'logistic_results.csv'
results_df.to_csv(results_path, index=False)
print('Saved results to:', results_path)

Saved results to: B:\PBCS\Major Project\BarrierLens_MP_G25_P48\outputs\stage1_results\logistic_results.csv
